# Velocity-Aware Denoising — kinematic_gamma sweep

Trains the U-Net with `L = α·MSE + β·(1−SSIM) + γ·|M1(pred) − M1(target)|`.

**Why:** every model measured so far degrades the GI wiggle (beam-only 0.920, dirty 0.891,
U-Net 0.804, DDRM 0.583). The U-Net optimises pixel accuracy; the wiggle is a sub-channel
velocity shift that per-channel smoothing destroys.

**Success = wiggle correlation above 0.804 WITHOUT losing M0/PSNR.** A model that preserves
velocity by not denoising is useless.

Trains and persists only. Wiggle scoring runs locally (`experiments/wiggle_all_methods.py`).

## 0. Bootstrap

In [1]:
import os, sys, subprocess, glob, re

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'midterm-prep'
if ON_KAGGLE:
    REPO = '/kaggle/working/EXXA'; PKG = os.path.join(REPO, 'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                        'https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',
                    'pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks')); sys.path.insert(0, PKG)

    run_re = re.compile(r'run_\d+_\d+_rt_\d+', re.I)
    roots = {}
    for p in glob.glob('/kaggle/input/**/run_*', recursive=True):
        if os.path.isdir(p) and run_re.search(os.path.basename(p)):
            roots[os.path.dirname(p)] = roots.get(os.path.dirname(p), 0) + 1
    if not roots:
        raise FileNotFoundError('No run_<id>_<step>_rt_<pp> folders under /kaggle/input.')
    DATA_DIR = max(roots, key=roots.get)
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'):
        os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../Line Emission Data'
print('DATA_DIR:', DATA_DIR)

Cloning into '/kaggle/working/EXXA'...
Updating files: 100% (5456/5456), done.


DATA_DIR: /kaggle/input/datasets/krishanyadav333/line-emission-data/Line Emission Data


## 0b. Pull latest `src/`

In [2]:
import importlib
if ON_KAGGLE:
    subprocess.run(['git','-C','/kaggle/working/EXXA','fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C','/kaggle/working/EXXA','reset','--hard','origin/'+BRANCH], check=True)
    print(subprocess.run(['git','-C','/kaggle/working/EXXA','log','-1','--oneline'],
                         capture_output=True, text=True).stdout.strip())
for m in [m for m in list(sys.modules) if m.startswith('src.')]:
    importlib.reload(sys.modules[m])
print('src/ reloaded')

From https://github.com/KrishanYadav333/EXXA
 * branch            midterm-prep -> FETCH_HEAD


HEAD is now at d9919bc results: store notebook 08's 3 surviving checkpoints + notebook 12's k=3
d9919bc results: store notebook 08's 3 surviving checkpoints + notebook 12's k=3
src/ reloaded


## 1. Config

In [3]:
import time, csv, shutil
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data.cube_split import split_cubes
from src.data.fits_cube_dataset import FITSChannelDataset
from src.training.sweep import train_unet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# Line FWHM is ~37 channels at a typical bright spaxel (measured), so the stack must be wide
# enough for a moment-1 to mean anything. k=15 -> 31 channels.
K           = 15
N_CH        = 2*K + 1
TARGET_SIZE = 256
N_SAMPLES   = 100
GAMMAS      = [10.0]   # 0.0/0.1/1.0 already ran and persisted before a kernel death on
                        # this arm (platform kill, no Python traceback in our code, at
                        # epoch 4 of gamma=10). Rerunning only the missing arm.
NW          = 2 if torch.cuda.is_available() else 0

WINNER = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
              lr=8.196504330730313e-4, alpha=0.8877681051398497,
              sched_patience=8, batch_size=4)     # smaller batch: 31 channels per sample

CKPT_DIR = '../results/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'{device} | {N_CH}-channel stacks | gammas {GAMMAS}')

cuda | 31-channel stacks | gammas [10.0]


## 2. Data — matched channel stacks

In [4]:
train_cubes, val_cubes, holdout_cubes = split_cubes(
    data_dir=DATA_DIR, n_holdout=3, val_fraction=0.2, seed=SEED)

_kw = dict(n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
           subtract_continuum=True, continuum_n=5,
           n_neighbors=K, stack_target=True, verbose=False)
train_ds = FITSChannelDataset(train_cubes, **_kw)
val_ds   = FITSChannelDataset(val_cubes, **_kw)

d, c = train_ds[0]
print(f'train {len(train_ds)} | val {len(val_ds)}')
print(f'item: dirty {tuple(d.shape)}  clean {tuple(c.shape)}')
assert d.shape == c.shape == (N_CH, TARGET_SIZE, TARGET_SIZE), 'stack shapes must match'

# Channel velocities in km/s, needed by the loss. CDELT3 is km/s in these headers.
from astropy.io import fits
_h = fits.getheader(train_cubes[0]['clean'])
VELAX = (np.arange(N_CH) - K) * float(_h['CDELT3'])
print(f'velax: {VELAX[0]:+.3f} to {VELAX[-1]:+.3f} km/s (dv={float(_h["CDELT3"]):.4f})')

CUBE-LEVEL SPLIT (grouped by RunID — no channel-level leakage)
  data_dir          : /kaggle/input/datasets/krishanyadav333/line-emission-data/Line Emission Data
  total cubes       : 14  across 11 distinct RunIDs
  seed=42  n_holdout=3  val_fraction=0.2
----------------------------------------------------------------------
  TRAIN   :  7 cubes | RunIDs ['0006', '0010', '0020', '0022', '0030', '0035']
  VAL     :  2 cubes | RunIDs ['0016', '0036']
  HOLDOUT :  5 cubes | RunIDs ['0002', '0025', '0026']  <-- inference only, NEVER trained/validated
----------------------------------------------------------------------
  HOLDOUT cube folders (reserved for moment-map evaluation):
    - run_0002_00560_rt_00
    - run_0002_00560_rt_01
    - run_0002_00560_rt_04
    - run_0025_01000_rt_04
    - run_0026_00005_rt_04
train 700 | val 200
item: dirty (31, 256, 256)  clean (31, 256, 256)
velax: -1.500 to +1.500 km/s (dv=0.1000)


## 3. Sweep gamma

In [5]:
ROWS_CSV = os.path.join('../results', 'kinematic_gamma_sweep.csv')
FIELDS = ['gamma','psnr','ssim','mse','best_val_loss','best_epoch','epochs_run','wall_time_s']
rows = []

for gamma in GAMMAS:
    name = f'kin_gamma{gamma:g}'
    ckpt = os.path.join(CKPT_DIR, f'{name}.pth')
    print(f'\n{"="*70}\n=== {name}  (gamma={gamma})\n{"="*70}', flush=True)

    res = train_unet(train_ds, val_ds, device, **WINNER,
                     n_neighbors=K, out_channels=N_CH,
                     kinematic_gamma=gamma, velax_kms=VELAX,
                     min_epochs=15, max_epochs=45, patience=6,
                     num_workers=NW, seed=SEED, ckpt_path=ckpt, verbose=True)

    row = {'gamma': gamma, **{k: res[k] for k in FIELDS if k in res}}
    rows.append(row)
    with open(ROWS_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if f.tell() == 0: w.writeheader()
        w.writerow({k: row.get(k, '') for k in FIELDS})

    # RULES.md #1: persist the moment it exists, not in a cleanup cell.
    if ON_KAGGLE:
        shutil.copy2(ckpt, f'/kaggle/working/{name}.pth')
        print(f'  persisted -> /kaggle/working/{name}.pth')

    # Nothing freed GPU memory between arms before this; by the last arm, three prior
    # models/optimizers/dataloaders could still have memory lingering. Caught a kernel death
    # on gamma=10 with no Python traceback in this code, consistent with an OOM kill. This
    # doesn't change any result, only what's resident on the GPU going into the next arm.
    del res
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


=== kin_gamma10  (gamma=10.0)
  ep   1 | train 0.3726 | val 0.3816 | lr 8.2e-04 (128s) *best
  ep   2 | train 0.2467 | val 0.2116 | lr 8.2e-04 (99s) *best
  ep   3 | train 0.2005 | val 0.1643 | lr 8.2e-04 (99s) *best
  ep   4 | train 0.1859 | val 0.2698 | lr 8.2e-04 (98s)
  ep   5 | train 0.2063 | val 0.2507 | lr 8.2e-04 (99s)
  ep   6 | train 0.1606 | val 0.1364 | lr 8.2e-04 (100s) *best
  ep   7 | train 0.1395 | val 0.1057 | lr 8.2e-04 (99s) *best
  ep   8 | train 0.1340 | val 0.1604 | lr 8.2e-04 (100s)
  ep   9 | train 0.1313 | val 0.1012 | lr 8.2e-04 (98s) *best
  ep  10 | train 0.1163 | val 0.0896 | lr 8.2e-04 (100s) *best
  ep  11 | train 0.1060 | val 0.1244 | lr 8.2e-04 (96s)
  ep  12 | train 0.1084 | val 0.0994 | lr 8.2e-04 (99s)
  ep  13 | train 0.1037 | val 0.1035 | lr 8.2e-04 (97s)
  ep  14 | train 0.1011 | val 0.1842 | lr 8.2e-04 (98s)
  ep  15 | train 0.2688 | val 0.2939 | lr 8.2e-04 (98s)
  ep  16 | train 0.1984 | val 0.2123 | lr 8.2e-04 (99s)
  early stop at epoch 16 (n

## 4. Results

In [6]:
print(f"{'gamma':>7s} {'PSNR':>8s} {'SSIM':>8s} {'val_loss':>10s} {'epochs':>7s}")
for r in rows:
    print(f"{r['gamma']:7g} {r.get('psnr',float('nan')):8.3f} {r.get('ssim',float('nan')):8.4f} "
          f"{r.get('best_val_loss',float('nan')):10.5f} {r.get('epochs_run',''):>7}")

print('\nPSNR here is over the full 31-channel stack, so it is NOT comparable to notebook')
print('05\'s single-channel numbers. Compare gammas against each other, and take the wiggle')
print('result from the local scoring script.')

if len(rows) > 1:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    g = [r['gamma'] for r in rows]
    ax[0].plot(g, [r.get('psnr', np.nan) for r in rows], 'o-')
    ax[0].set_xlabel('kinematic_gamma'); ax[0].set_ylabel('PSNR (31-ch stack)')
    ax[0].set_xscale('symlog', linthresh=0.1); ax[0].grid(alpha=0.3)
    ax[1].plot(g, [r.get('best_val_loss', np.nan) for r in rows], 'o-', color='crimson')
    ax[1].set_xlabel('kinematic_gamma'); ax[1].set_ylabel('best val loss')
    ax[1].set_xscale('symlog', linthresh=0.1); ax[1].grid(alpha=0.3)
    ax[1].set_title('not comparable across gamma: different objectives')
    plt.tight_layout(); plt.savefig('../results/kinematic_gamma_sweep.png', dpi=130); plt.show()

  gamma     PSNR     SSIM   val_loss  epochs
     10   24.333   0.9293    0.08960      16

PSNR here is over the full 31-channel stack, so it is NOT comparable to notebook
05's single-channel numbers. Compare gammas against each other, and take the wiggle
result from the local scoring script.


## 5. Next

Download the `kin_gamma*.pth` checkpoints, then score locally:

```
PYTHONPATH=.. python3 experiments/wiggle_all_methods.py
```

Success is wiggle correlation above the U-Net's **0.804** while M0 and PSNR hold up.

In [7]:
from src.evaluation.collect_outputs import collect_outputs
collect_outputs('08-kinematic-loss', ['*.png', '*.csv', '*.pth'])

collected 157 file(s), 175.4 MiB -> /kaggle/working/outputs/08-kinematic-loss/2026-09-10T135734_d9919bc
       3.64 MiB  1.png
       0.71 MiB  10.png
       0.07 MiB  11.png
       0.03 MiB  12.png
       0.67 MiB  13.png
       0.05 MiB  14.png
       0.07 MiB  15.png
       0.10 MiB  16.png
       1.78 MiB  17.png
       0.83 MiB  18.png
       0.06 MiB  19.png
       0.45 MiB  2.png
       0.03 MiB  3.png
       1.37 MiB  4.png
       0.04 MiB  5.png
       0.24 MiB  6.png
       0.13 MiB  7.png
       0.05 MiB  8.png
       1.62 MiB  9.png
       0.36 MiB  Exoplanet_WASP-96_b_NIRISS_transmission_spectrum_article.png
       0.29 MiB  Exoplanet_WASP-96_b_transit_light_curve_article.png
       0.30 MiB  V12_channel100.png
       0.05 MiB  V12_holdout_summary_chart.png
       0.04 MiB  V12_loss_curve.png
       0.96 MiB  V12_moment_comparison.png
       0.82 MiB  V12_moment_maps.png
       0.89 MiB  V12_val_channels.png
       0.27 MiB  V7_channel100.png
       0.04 MiB  V7_loss_curve

'/kaggle/working/outputs/08-kinematic-loss/2026-09-10T135734_d9919bc'